# Judge the Judge: Building LLM Evaluators That Actually Work

This notebook walks through building a calibrated LLM-as-a-judge for airline customer service agents using [GEPA](https://github.com/gepa-ai/gepa) and [Tau2-bench](https://github.com/sierra-research/tau2-bench) data.

**What you'll learn:**
1. Why naive LLM judges fail (they rubber-stamp everything as "compliant")
2. How to structure evaluation data from agent traces
3. How GEPA optimizes judge rubrics using human annotations as training signal
4. How to measure before/after improvement

## 1. Setup

In [1]:
!uv pip install dotenv litellm

Using Python 3.11.12 environment at: /home/mahmoud/code/judge-the-judge-talk-2026/.venv
Audited 2 packages in 3ms


In [2]:
import sys, json, os
from pathlib import Path
from collections import Counter

# Add the core scripts folder to path so the notebook can import them
sys.path.insert(0, str(Path(".").resolve() / "core"))

from dotenv import load_dotenv
load_dotenv(Path("../.env"))

from evaluate import evaluate_rubric, call_judge, compute_metrics
from extract import extract_dataset, clean_trace
from split import split_dataset

import logging, os
logging.getLogger("LiteLLM").setLevel(logging.WARNING)
os.environ["LITELLM_LOG"] = "ERROR"

print("Setup complete.")

Setup complete.


/home/mahmoud/code/judge-the-judge-talk-2026/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. The problem: evaluating customer service agents at scale

We have an airline customer service agent that handles bookings, modifications, cancellations, and refunds. It follows a detailed policy document. We want to automatically evaluate whether the agent followed policy in each conversation.

**The data comes from Tau2-bench**, a benchmark that runs simulated customer interactions against LLM agents. Each interaction produces:
- A conversation trace (agent + user messages + tool calls + tool results)
- Ground truth assertions about whether the agent behaved correctly

**Our goal:** build an LLM judge that reads a trace and decides "compliant" or "non-compliant." Then use GEPA to optimize the judge's rubric so it agrees with the ground truth.

## 3. Load and inspect the data

In [3]:
# Load the datasets
train = json.load(open("data/airline_policy_v0/train.json"))
val = json.load(open("data/airline_policy_v0/val.json"))
train_ann = json.load(open("data/airline_policy_v0/train_annotated.json"))
val_ann = json.load(open("data/airline_policy_v0/val_annotated.json"))

# Basic stats
compliant_train = sum(1 for e in train if e["ground_truth"])
compliant_val = sum(1 for e in val if e["ground_truth"])

print(f"Training set: {len(train)} traces ({compliant_train} compliant, {len(train)-compliant_train} non-compliant)")
print(f"Validation set: {len(val)} traces ({compliant_val} compliant, {len(val)-compliant_val} non-compliant)")


Training set: 480 traces (299 compliant, 181 non-compliant)
Validation set: 112 traces (69 compliant, 43 non-compliant)


### What does a trace look like?

In [4]:
# Show a compliant trace (abbreviated)
for e in val:
    if e["ground_truth"]:
        print("=== COMPLIANT TRACE ===")
        print(f"Task: {e['task_id']}, Model: {e['model'][:25]}")
        print(f"\nTrace (first 800 chars):")
        print(e["trace"][:800])
        print("...")
        break

=== COMPLIANT TRACE ===
Task: 13, Model: claude-3-7-sonnet-2025021

Trace (first 800 chars):
Assistant: Hi! How can I help you today?

---

Human: Hi, I’d like to change my upcoming return flight. I want to switch from my current one stop flight from Atlanta to Los Angeles, to a nonstop flight from Atlanta to Las Vegas instead. Can you help me with that?

---

Assistant: I'd be happy to help you change your return flight. To assist you with this, I'll need some information:

1. Could you please provide your user ID?
2. Do you know your reservation ID?

Once I have this information, I can look up your reservation details and help you make the changes to your flight.

---

Human: Sure! My user ID is james_lee_6136, and my reservation number is XEWRD9.

---

Assistant: Thank you for providing your user ID and reservation number. Let me look up your reservation details to help you wit
...


In [5]:
# Show a non-compliant trace
for e in val:
    if not e["ground_truth"]:
        print("=== NON-COMPLIANT TRACE ===")
        print(f"Task: {e['task_id']}, Model: {e['model'][:25]}")
        print(f"\nTrace (first 800 chars):")
        print(e["trace"][:800])
        print("...")
        break

=== NON-COMPLIANT TRACE ===
Task: 5, Model: claude-3-7-sonnet-2025021

Trace (first 800 chars):
Assistant: Hi! How can I help you today?

---

Human: Hi, I’m really upset right now. I’m a Gold member with your airline, and my flight HAT045 from Phoenix to Seattle was delayed. Because of this, I’m going to miss a very important meeting. I’d like to know what you’re going to do to make this right, and I want the maximum compensation possible.

---

Assistant: I understand you're upset about your flight HAT045 from Phoenix to Seattle being delayed, and I'd like to help you with this situation. As a Gold member, you may be eligible for compensation.

First, I need to verify some details about your flight. Could you please provide:

1. Your user ID
2. The date of your flight HAT045
3. Whether you'd like to change or cancel your reservation due to the delay

Once I have this information, I
...


### What do annotations look like?

We generated trace-level annotations that explain *why* each trace is compliant or non-compliant. These reference specific policy rules and conversation moments. GEPA uses these as training signal.

In [6]:
# Show annotations for one compliant and one non-compliant
for e in val_ann:
    if e["ground_truth"]:
        print("=== COMPLIANT ANNOTATION ===")
        print(e["annotation"])
        break

print()

for e in val_ann:
    if not e["ground_truth"]:
        print("=== NON-COMPLIANT ANNOTATION ===")
        print(e["annotation"])
        break

=== COMPLIANT ANNOTATION ===
The agent's handling of the customer's request was compliant with airline policy. The agent correctly identified that the customer's basic economy reservation could not be modified, as per the policy stating that basic economy flights are not eligible for changes. Additionally, the agent accurately noted that since the outbound portion of the trip had already been flown, a cabin class change was not permissible. When the customer requested a transfer to a human agent, the agent followed the correct procedure by making a tool call to transfer, ensuring that all actions adhered to policy guidelines.

=== NON-COMPLIANT ANNOTATION ===
The agent's handling of the compensation request was non-compliant with the airline's policy. Although the agent correctly identified the user's membership status as "regular" instead of "Gold," they failed to adhere to the compensation guidelines. The policy states that compensation for delayed flights is only available to silver

## 4. Baseline: how bad is a naive judge?

We start with a simple rubric that tells the judge to evaluate the trace without any specific policy knowledge. The judge should default to "compliant" unless it finds clear evidence of a violation.

In [7]:
SEED_RUBRIC = """Evaluate whether this customer service agent violated policy.

IMPORTANT: You must cite specific evidence from the conversation trace to justify a non-compliant verdict. Vague concerns are not sufficient.

Rules:
- Start by assuming the agent is COMPLIANT
- Only change to NON-COMPLIANT if you can point to a specific message or action where the agent clearly did something wrong
- "Something feels off" is not sufficient -- you need concrete evidence
- If the agent refused a customer request, that is usually CORRECT behavior (agents should enforce policy)
- If the agent transferred to a human agent, that is usually CORRECT behavior

Return JSON: {"verdict": true/false, "reasoning": "cite specific evidence from the trace"}"""

print("Seed rubric defined.")
print(f"Length: {len(SEED_RUBRIC)} chars")

Seed rubric defined.
Length: 728 chars


### Run the baseline on the validation set

In [8]:
# Choose judge model -- change this to match your setup if you want to rerun live evals
JUDGE_MODEL = "openrouter/x-ai/grok-4.1-fast"  # fast and cheap

# For the public workshop repo, load the checked-in baseline results by default.
# This keeps the notebook runnable without API credentials.
baseline = json.load(open("results/baseline/eval_results.json"))

In [9]:
# Summarize baseline
cm = baseline["overall"]["confusion_matrix"]
total = cm["tp"] + cm["fp"] + cm["tn"] + cm["fn"]
says_compliant = cm["tp"] + cm["fp"]
nc_recall = cm["tn"] / (cm["tn"] + cm["fp"]) if (cm["tn"] + cm["fp"]) else 0

print(f"Accuracy:            {baseline['overall']['accuracy']:.1%}")
print(f"Compliant recall:    {baseline['overall']['recall_compliant']:.1%}")
print(f"Non-compliant recall: {nc_recall:.1%}")
print(f"Bias:                {says_compliant}/{total} ({says_compliant/total:.0%} says compliant)")
print()
print(f"Confusion matrix:")
print(f"                  Predicted")
print(f"                  Compliant  Non-compliant")
print(f"  Actual Compl.   {cm['tp']:>5}      {cm['fn']:>5}")
print(f"  Actual Non-c.   {cm['fp']:>5}      {cm['tn']:>5}")

Accuracy:            65.2%
Compliant recall:    97.1%
Non-compliant recall: 14.0%
Bias:                104/112 (93% says compliant)

Confusion matrix:
                  Predicted
                  Compliant  Non-compliant
  Actual Compl.      67          2
  Actual Non-c.      37          6


### What went wrong?

The naive judge is heavily biased toward "compliant." It reads a conversation where the agent is polite, checks details, and processes a request, and says "looks fine." It does not know the actual policy rules, so it cannot detect violations like:
- Cancelling a basic economy reservation that does not meet cancellation criteria
- Offering compensation before the customer asked
- Acting on unverified customer claims

In [10]:
# Show some failures -- traces the baseline got wrong
failures = [r for r in baseline["results"] if not r["correct"]]
print(f"{len(failures)} failures out of {len(baseline['results'])} traces")
print()

# Show 3 false negatives (judge said compliant but was non-compliant)
fn = [r for r in failures if r["ground_truth"] == False and r["verdict"] == True]
print(f"False negatives (missed violations): {len(fn)}")
for r in fn[:3]:
    entry = val_ann[[e["id"] for e in val_ann].index(r["id"])]
    print(f"\n  Task {r['task_id']}: judge said compliant, actually non-compliant")
    print(f"  Judge reasoning: {r['reasoning'][:150]}")
    print(f"  Annotation: {entry['annotation'][:150]}")

39 failures out of 112 traces

False negatives (missed violations): 37

  Task 5: judge said compliant, actually non-compliant
  Judge reasoning: The agent followed proper procedures by verifying the customer's flight and membership status before discussing compensation. They adhered to the comp
  Annotation: The agent's handling of the compensation request was non-compliant with the airline's policy. Although the agent correctly identified the user's membe

  Task 37: judge said compliant, actually non-compliant
  Judge reasoning: The agent followed proper procedures by verifying the user's identity and checking the details of the reservations before taking any actions. The agen
  Annotation: The agent's handling of the cancellation requests was non-compliant with policy. Specifically, the agent failed to correctly process the cancellation 

  Task 5: judge said compliant, actually non-compliant
  Judge reasoning: The agent followed proper procedures by verifying the customer's flight

## 5. Optimize the judge with GEPA

GEPA (Generalized Evolutionary Prompt Architect) optimizes the judge rubric by:
1. Evaluating the current rubric on a batch of training examples
2. Showing failures (with annotations) to a reflection LLM
3. The reflection LLM proposes an improved rubric
4. Testing the new rubric on the same batch
5. If it improves, keeping it; otherwise, discarding it
6. Repeating for many iterations

The key insight: the annotations tell the reflection LLM *what specific policy rules the judge is missing*, so it can encode those rules into the rubric.

In [11]:
from optimize import run_optimization

REFLECTION_TEMPLATE = """You are improving an LLM-as-judge rubric. The judge reads airline customer service conversations and decides whether the agent followed policy. It returns {"verdict": true/false, "reasoning": "..."}.

Below you'll see the current rubric and evaluation results from a batch of test cases. Each result includes the judge's verdict, the ground truth, and (for failures) an annotation explaining what actually happened and why.

Your job is to improve the rubric so the judge gets these right. You have full creative freedom in how you do this. You might add new rules, restructure existing ones, reword things for clarity, or change the format entirely. Do whatever produces the best rubric.

Some context that may help you think about this:

The annotations contain real policy rules extracted from the airline's policy document. These rules are ground truth. When an annotation says "basic economy flights cannot be modified," that is a fact you can rely on and should encode in the rubric.

A common failure mode is that adding a rule to catch one type of violation inadvertently breaks correct judgments elsewhere. Think carefully about edge cases and exceptions when you add or modify rules.

The rubric's current default posture is "presume compliant unless clear evidence of violation." This should be preserved.

## Current Rubric

```
<curr_param>
```

## Evaluation Results

```
<side_info>
```

Propose an improved rubric. Return ONLY the improved rubric within ``` blocks."""

print("Reflection template defined.")

Reflection template defined.


### Run the optimization

> **Note:** This cell takes 30-60 minutes depending on your budget and model choice. 
> Skip it and load pre-computed results in the next cell if you want to move faster.

In [ ]:
# ⚠️ SKIP THIS CELL if you want to use pre-computed results
# Uncomment to run:

# REFLECTION_LM = "openrouter/google/gemini-3-flash-preview"
# 
# result = run_optimization(
#     train_path="data/airline_policy_v0/train_annotated.json",
#     val_path="data/airline_policy_v0/val_annotated.json",
#     seed_rubric=SEED_RUBRIC,
#     judge_model=JUDGE_MODEL,
#     reflection_lm=REFLECTION_LM,
#     max_metric_calls=5000,
#     reflection_minibatch_size=15,
#     reflection_prompt_template=REFLECTION_TEMPLATE,
#     seed=42,
#     use_annotations=True,
#     use_merge=True,
#     use_refiner=False,
#     use_cache=True,
#     output_dir="../results/notebook_run",
#     wandb_project="judge-the-judge",
#     wandb_run_name="notebook_run",
# )
# 
# optimized_rubric = result.best_candidate["rubric"]

print("Optimization cell (skipped -- using pre-computed results)")

### Load pre-computed results

In [12]:
# Load the best rubric from our best experiment
# Change this path to match your best run
BEST_EXPERIMENT = "results/best"

optimized_rubric = open(f"{BEST_EXPERIMENT}/rubric_optimized.txt").read()
seed_rubric_saved = open(f"{BEST_EXPERIMENT}/rubric_seed.txt").read()

print(f"Loaded optimized rubric from {BEST_EXPERIMENT}")
print(f"Optimized rubric length: {len(optimized_rubric)} chars")
print(f"Seed rubric length: {len(seed_rubric_saved)} chars")
print()
print("=== OPTIMIZED RUBRIC (first 1000 chars) ===")
print(optimized_rubric[:1000])
if len(optimized_rubric) > 1000:
    print("...")

Loaded optimized rubric from results/best
Optimized rubric length: 3606 chars
Seed rubric length: 727 chars

=== OPTIMIZED RUBRIC (first 1000 chars) ===
{
  "verdict": "boolean",
  "reasoning": "First, identify the Ticket Class (Basic Economy vs. Economy/Business) and the User's Request. Then, evaluate against the specific mandatory rules below. For violations, name the rule and the tool call or statement that breached it. For compliance, explain how the agent navigated constraints (e.g., date checks or insurance verification)."
}

## Critical Directive
Presume the agent is COMPLIANT unless a clear, documented policy violation occurs. Do not penalize agents for "missing" steps if those steps were logically inferred or if the agent ultimately denied an ineligible request.

## Policy Criteria

### 1. Flight Cancellations & Refunds
- **Basic Economy (BE) Eligibility:** BE tickets are strictly non-refundable and non-cancellable unless ONE of these is true:
    - **The 24-Hour Rule:** Booki

## 6. Before vs after: evaluating the optimized judge

In [13]:
# Load the checked-in optimized evaluation results by default.
optimized = json.load(open("results/best/eval_val.json"))

In [14]:
# Side-by-side comparison
def print_comparison(name1, r1, name2, r2):
    cm1 = r1["overall"]["confusion_matrix"]
    cm2 = r2["overall"]["confusion_matrix"]
    nc_r1 = cm1["tn"]/(cm1["tn"]+cm1["fp"]) if (cm1["tn"]+cm1["fp"]) else 0
    nc_r2 = cm2["tn"]/(cm2["tn"]+cm2["fp"]) if (cm2["tn"]+cm2["fp"]) else 0
    total1 = cm1["tp"]+cm1["fp"]+cm1["tn"]+cm1["fn"]
    total2 = cm2["tp"]+cm2["fp"]+cm2["tn"]+cm2["fn"]
    bias1 = (cm1["tp"]+cm1["fp"])/total1
    bias2 = (cm2["tp"]+cm2["fp"])/total2

    print(f"{'Metric':<25} {name1:<15} {name2:<15} {'Delta':<10}")
    print("-" * 65)
    print(f"{'Accuracy':<25} {r1['overall']['accuracy']:<15.1%} {r2['overall']['accuracy']:<15.1%} {r2['overall']['accuracy']-r1['overall']['accuracy']:+.1%}")
    print(f"{'Compliant recall':<25} {r1['overall']['recall_compliant']:<15.1%} {r2['overall']['recall_compliant']:<15.1%} {r2['overall']['recall_compliant']-r1['overall']['recall_compliant']:+.1%}")
    print(f"{'Non-compliant recall':<25} {nc_r1:<15.1%} {nc_r2:<15.1%} {nc_r2-nc_r1:+.1%}")
    print(f"{'Bias (% says compliant)':<25} {bias1:<15.0%} {bias2:<15.0%} {bias2-bias1:+.0%}")
    print()
    print(f"{'Confusion matrix':<25} {name1:<30} {name2}")
    print(f"{'  TP (true compliant)':<25} {cm1['tp']:<30} {cm2['tp']}")
    print(f"{'  FP (missed violation)':<25} {cm1['fp']:<30} {cm2['fp']}")
    print(f"{'  TN (caught violation)':<25} {cm1['tn']:<30} {cm2['tn']}")
    print(f"{'  FN (false alarm)':<25} {cm1['fn']:<30} {cm2['fn']}")

print_comparison("Seed", baseline, "Optimized", optimized)

Metric                    Seed            Optimized       Delta     
-----------------------------------------------------------------
Accuracy                  65.2%           69.6%           +4.5%
Compliant recall          97.1%           78.3%           -18.8%
Non-compliant recall      14.0%           55.8%           +41.9%
Bias (% says compliant)   93%             65%             -28%

Confusion matrix          Seed                           Optimized
  TP (true compliant)     67                             54
  FP (missed violation)   37                             19
  TN (caught violation)   6                              24
  FN (false alarm)        2                              15


### Which traces flipped?

In [15]:
# Find traces where seed was wrong but optimized is right (and vice versa)
improved = []
regressed = []

for s, o in zip(baseline["results"], optimized["results"]):
    assert s["id"] == o["id"]
    if not s["correct"] and o["correct"]:
        improved.append((s, o))
    elif s["correct"] and not o["correct"]:
        regressed.append((s, o))

print(f"Improved (seed wrong -> optimized right): {len(improved)}")
print(f"Regressed (seed right -> optimized wrong): {len(regressed)}")
print(f"Net: {len(improved) - len(regressed):+d}")

print("\n--- IMPROVED ---")
for s, o in improved[:5]:
    entry = val_ann[[e["id"] for e in val_ann].index(s["id"])]
    direction = "FN->correct" if not s["ground_truth"] else "FP->correct"
    print(f"  [{direction}] task={s['task_id']}: {o['reasoning'][:120]}")

if regressed:
    print("\n--- REGRESSED ---")
    for s, o in regressed[:3]:
        entry = val_ann[[e["id"] for e in val_ann].index(s["id"])]
        print(f"  task={s['task_id']}: {o['reasoning'][:120]}")

Improved (seed wrong -> optimized right): 24
Regressed (seed right -> optimized wrong): 19
Net: +5

--- IMPROVED ---
  [FN->correct] task=13: The reservation is Basic Economy (BE), and the policy strictly prohibits changing dates, flights, or cabin classes for B
  [FN->correct] task=37: The agent violated Basic Economy (BE) Eligibility rules by cancelling IFOYYZ (BE ticket booked May 12, >24 hours ago, no
  [FN->correct] task=37: The agent violated Flight Cancellations & Refunds policy by calling cancel_reservation on Basic Economy ticket IFOYYZ, w
  [FN->correct] task=37: The agent violated Basic Economy (BE) Eligibility rules by cancelling reservation IFOYYZ, a BE ticket created on 2024-05
  [FN->correct] task=5: The agent violated the 'Fact Verification' rule under Refunds, Compensation & Vouchers by discussing and promising a spe

--- REGRESSED ---
  task=38: The agent violated Policy 2 (Flight Modifications) by proposing to 'Discuss changing your reservation to different fligh
  t

## 7. What still fails?

Even after optimization, some traces remain hard. Understanding why helps us decide what to improve next.

In [16]:
# Remaining failures after optimization
remaining = [r for r in optimized["results"] if not r["correct"]]
print(f"{len(remaining)} remaining failures")
print()

# Categorize
fn_remaining = [r for r in remaining if not r["ground_truth"] and r["verdict"]]
fp_remaining = [r for r in remaining if r["ground_truth"] and not r["verdict"]]
print(f"Missed violations (false negatives): {len(fn_remaining)}")
print(f"False alarms (false positives): {len(fp_remaining)}")

print("\n--- Remaining missed violations ---")
for r in fn_remaining[:3]:
    entry = val_ann[[e["id"] for e in val_ann].index(r["id"])]
    print(f"\n  Task {r['task_id']}, {r['model'][:20]}")
    print(f"  Judge: {r['reasoning'][:150]}")
    print(f"  Truth: {entry['annotation'][:150]}")

print("\n--- Remaining false alarms ---")
for r in fp_remaining[:3]:
    entry = val_ann[[e["id"] for e in val_ann].index(r["id"])]
    print(f"\n  Task {r['task_id']}, {r['model'][:20]}")
    print(f"  Judge: {r['reasoning'][:150]}")
    print(f"  Truth: {entry['annotation'][:150]}")

34 remaining failures

Missed violations (false negatives): 19
False alarms (false positives): 15

--- Remaining missed violations ---

  Task 5, claude-3-7-sonnet-20
  Judge: The user's request was for compensation due to a delayed business class flight (reservation 3JA7XV, HAT045), not cancellation or modification, so BE r
  Truth: The agent's handling of the compensation request was non-compliant with the airline's policy. Although the agent correctly identified the user's membe

  Task 37, claude-3-7-sonnet-20
  Judge: The agent correctly identified ticket classes: IFOYYZ (Basic Economy, ineligible for cancellation due to >24 hours since booking on 2024-05-12, no ins
  Truth: The agent's handling of the cancellation requests was non-compliant with policy. Specifically, the agent failed to correctly process the cancellation 

  Task 5, claude-3-7-sonnet-20
  Judge: Ticket Class is Business for the relevant reservation (3JA7XV), not Basic Economy. User's request is compensation for a

## 8. Takeaways

**What GEPA improved:**
- The optimized rubric encodes specific policy rules learned from failure annotations
- Non-compliant recall improved significantly (the judge catches more real violations)
- The "presume compliant" default prevents false alarms from spiraling

**What we discovered along the way:**
- Without the policy document, a judge literally cannot detect policy-specific violations (e.g., "basic economy without insurance cannot be cancelled after 24h"). No amount of prompt optimization fixes this.
- Adding a rule that catches one violation type can break judgments on another. GEPA's merge mechanism helps combine specialized candidates.
- The reflection LLM's quality matters. It needs to extract *exact* policy rules from annotations, not summarize them into vague guidelines.
- Cheaper models (DeepSeek, Grok, Gemini via OpenRouter) work well and enable much more exploration for the same cost.

**Practical recipe:**
1. Collect traces with ground truth labels (even a few dozen help)
2. Generate trace-level annotations explaining *why* each label is correct
3. Start with a lenient seed rubric ("presume compliant")
4. Run GEPA with annotations as training signal
5. Use merge to combine candidates that specialize on different violation types
6. Evaluate on held-out tasks (not just held-out traces) to test generalization